# OptiVision 0.6.2: GPU checks (the two fixes, then other model families)

Run on a Kaggle (or Colab) GPU with Internet on, then Run all. It installs 0.6.2 from the `fix-0.6.2` branch and,
for each model in a list, runs one self-contained script in its own process (so a failure or an out-of-memory
in one model cannot touch the next): build a small folder, search, `check()`, export to Qdrant with the
client and collection passed positionally, then `add`, reload and search again. Every reload must print no
"The model now differs from the build record" warning. The first model also exports to a real Qdrant server.
Each model prints PASS or FAIL and the last cell prints a summary. A FAIL is a finding.

In [ ]:
%pip install -q -U "optivision-rag[easy-st,qdrant,corpus] @ git+https://github.com/Daemon-VI/optivision-rag@fix-0.6.2"
%pip uninstall -y -q torchao
import optivision
print("optivision", optivision.__version__)

In [ ]:
import json, pathlib, subprocess, sys, shutil, time, urllib.request

RESULTS = {}
d = pathlib.Path("b_pdfs")
d.mkdir(exist_ok=True)
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from PIL import Image, ImageDraw, ImageFont


def make(folder, name, pages):
    c = canvas.Canvas(str(folder / name), pagesize=A4)
    for title, body in pages:
        c.setFont("Helvetica-Bold", 20)
        c.drawString(60, 780, title)
        c.setFont("Helvetica", 12)
        y = 740
        for line in body:
            c.drawString(60, y, line)
            y -= 20
        c.showPage()
    c.save()


make(d, "finance.pdf", [
    ("Quarterly revenue by region", ["North: 4.2 million", "South: 3.1 million", "East: 5.7 million", "West: 2.9 million"]),
    ("Operating costs", ["Salaries are 62 percent of costs", "Cloud hosting is 14 percent", "Marketing is 9 percent"]),
    ("Risk register", ["Supplier concentration is the largest risk", "Currency exposure is hedged"]),
])
make(d, "garden.pdf", [
    ("Growing tomatoes", ["Plant after the last frost", "Water deeply twice a week", "Stake the vines early"]),
    ("Composting basics", ["Mix green kitchen scraps with brown leaves", "Turn the pile every two weeks"]),
])
img = Image.new("RGB", (1240, 1754), "white")
ImageDraw.Draw(img).text((120, 200), "NOTICE: The warehouse closes at 6 pm on Fridays.", fill="black",
                         font=ImageFont.load_default(size=44))
img.save(d / "notice.pdf")
t = pathlib.Path("t_pdfs")
t.mkdir(exist_ok=True)
for n in ("finance.pdf", "garden.pdf"):
    shutil.copy(d / n, t / n)
print(sorted(p.name for p in d.iterdir()), sorted(p.name for p in t.iterdir()))

Start a real Qdrant server (the official release binary) for the first model.

In [ ]:
subprocess.run("wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O q.tgz && tar xzf q.tgz", shell=True, check=True)
qproc = subprocess.Popen(["./qdrant"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(40):
    try:
        urllib.request.urlopen("http://localhost:6333/readyz")
        print("qdrant server up")
        break
    except Exception:
        time.sleep(1)

In [ ]:
%%writefile sub.py
import sys, json, pathlib, traceback
model, server, kind = sys.argv[1], sys.argv[2] == "server", sys.argv[3]
slug = model.replace("/", "_")
res = {"model": model, "steps": {}}
folder = "t_pdfs" if kind == "text" else "b_pdfs"
pathlib.Path("out").mkdir(exist_ok=True)
out = "out/" + slug + ".ovx"


def step(name, fn):
    try:
        res["steps"][name] = ["PASS", str(fn())[:300]]
    except Exception as exc:
        res["steps"][name] = ["FAIL", type(exc).__name__ + ": " + str(exc)[:400]]
        traceback.print_exc()


import optivision.easy as ov
S = {}


def build():
    S["idx"] = ov.build(folder, model=model, target=0.97, path=out, work_dir=None, progress=False)
    return "built"


def search():
    wants = {"Which region had the highest revenue?": "finance.pdf", "How often should I water tomato plants?": "garden.pdf"}
    if kind != "text":
        wants["When does the warehouse close on Friday?"] = "notice.pdf"
    bad = []
    for q, f in wants.items():
        top = S["idx"].search(q, k=1)[0]
        if f not in top.id:
            bad.append(q + " -> " + top.id)
    assert not bad, "; ".join(bad)
    return str(len(wants)) + "/" + str(len(wants)) + " top hits right"


def check():
    c = S["idx"].check()
    assert c.record.matches, "build record reports a difference: " + str(c.record)
    return "record matches"


def export_positional():
    from qdrant_client import QdrantClient
    cl = QdrantClient(":memory:")
    S["idx"].export_qdrant(cl, "pages")
    n = cl.count("pages").count
    assert n > 0, "no points written"
    return "points=" + str(n)


def export_server():
    from qdrant_client import QdrantClient
    S["idx"].export_qdrant("http://localhost:6333", "pages_" + slug, recreate=True)
    return "points=" + str(QdrantClient("http://localhost:6333").count("pages_" + slug).count)


def add_reload():
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas
    c = canvas.Canvas(folder + "/travel.pdf", pagesize=A4)
    c.setFont("Helvetica", 14)
    c.drawString(60, 780, "Museum hours")
    c.drawString(60, 750, "The city museum opens at 9 am and is closed on Mondays")
    c.showPage()
    c.save()
    S["idx"].add(folder)
    S["idx"].close()
    S.clear()
    import gc
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    idx2 = ov.load(out)
    top = idx2.search("Which day is the museum closed?", k=1)[0]
    assert "travel.pdf" in top.id, "added file not found: " + top.id
    assert idx2.record_differences == [], "false record difference after reload: " + str(idx2.record_differences)
    return "added file found; reload printed no record difference"


step("build", build)
if "idx" in S:
    step("search", search)
    step("check", check)
    step("export positional", export_positional)
    if server:
        step("export real server", export_server)
    step("add + reload, no false warning", add_reload)
print("RESULT " + json.dumps(res))

In [ ]:
def run_models(models):
    for model, server, kind in models:
        t0 = time.time()
        try:
            p = subprocess.run([sys.executable, "sub.py", model, "server" if server else "no", kind],
                               capture_output=True, text=True, timeout=2400)
            line = [ln for ln in p.stdout.splitlines() if ln.startswith("RESULT ")]
            if line:
                RESULTS[model] = json.loads(line[-1][7:])["steps"]
            else:
                RESULTS[model] = {"process": ["FAIL", (p.stderr or p.stdout)[-600:]]}
        except Exception as exc:
            RESULTS[model] = {"process": ["FAIL", type(exc).__name__ + ": " + str(exc)[:300]]}
        ok = all(v[0] == "PASS" for v in RESULTS[model].values())
        print("PASS" if ok else "FAIL", model, round(time.time() - t0), "s")
        for k, v in RESULTS[model].items():
            print("   ", v[0], k, "|", v[1][:200])


# group A: sentence-transformers loaders (the first model also goes to the real Qdrant server)
run_models([
    ("vidore/colqwen2-v1.0-hf", True, "image"),
    ("vidore/colpali-v1.3-hf", False, "image"),
    ("Hcompany/NeoMME-260M-Retriever-ST-late", False, "image"),
    ("answerdotai/answerai-colbert-small-v1", False, "text"),
])

Group B uses the colpali-engine loaders; installed after group A so a changed transformers version cannot affect group A. The second model's recorded run spread it over two GPUs, as this machine can.

In [ ]:
%pip install -q -U "optivision-rag[easy-colpali] @ git+https://github.com/Daemon-VI/optivision-rag@fix-0.6.2"
%pip uninstall -y -q torchao

In [ ]:
run_models([
    ("vidore/colSmol-500M", False, "image"),
    ("vidore/colqwen2.5-v0.2", False, "image"),
])

## Summary

In [ ]:
qproc.terminate()
for model, steps in RESULTS.items():
    fails = [k for k, v in steps.items() if v[0] != "PASS"]
    print("PASS" if not fails else "FAIL", model, fails)
print(json.dumps(RESULTS, indent=1)[:6000])